### Silver - Pedidos (cabeçalho e itens)

Notebook com o maior volume de transformações do projeto, por concentrar
o dado transacional central.

**Cabeçalho:**
- `order_id` em caixa baixa em parte das linhas
- 3 `order_id` duplicados com conteúdo conflitante:
  - `O00121`: uma versão com `order_date = 2025/03/01` (válida), outra
    com `2025-13-40` (mês 13, inexistente)
  - `O00081`: uma versão com `gross_amount = 305,52`, outra com `N/A`
  - `O00011`: duplicata com conteúdo idêntico
- `status_order` com `Faturado`/`faturado`/`EM_SEPARACAO`/`em separacao`/
  `entregue`/`cancelado`, além de registros vazios
- valores numéricos com separador decimal de ponto, de vírgula, ou
  literal `N/A`
- `net_amount` não reconcilia com `gross_amount - discount_amount` em
  parte dos pedidos (diferenças de centavos a aproximadamente R$120).
  O valor original é preservado como referência de receita, com uma
  flag de inconsistência para análise posterior
- `payment_details` é um JSON serializado como string

**Itens:**
- mesmo problema de caixa em `order_id`
- `unit_price` com vírgula decimal em parte dos registros
- `item_status` com `Ativo`/`ativo`/`cancelado`/vazio
- `product_code = P8888` sem cadastro correspondente (ver notebook de
  produtos); o dado é preservado nesta camada e tratado na Gold

In [ ]:
%run "../00_setup/00_setup_ambiente"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_cab_bronze = spark.table(f"{schema_name}.bronze_pedidos_cabecalho")
df_itens_bronze = spark.table(f"{schema_name}.bronze_pedidos_itens")

In [ ]:
df_cab_step1 = (
    df_cab_bronze
    .withColumn("order_id", F.upper(F.trim(F.col("order_id"))))
    .withColumn("customer_code", F.upper(F.trim(F.col("customer_code"))))
)

payment_schema = "struct<priority:string, source:string>"

df_cab_step2 = (
    df_cab_step1
    .withColumn("payment_parsed", F.from_json(F.col("payment_details"), payment_schema))
    .withColumn("order_priority", F.col("payment_parsed.priority"))
    .withColumn("order_source", F.col("payment_parsed.source"))
    .drop("payment_parsed", "payment_details")
)

A análise inicial de padrões de data, por substituição de dígitos por "#",
identificou apenas 2 formatos aparentes em order_date (`#-#-#` e `#/#/#`),
levando à suposição de que o padrão com "/" correspondesse sempre a
yyyy/MM/dd. A validação por amostragem revelou que esse padrão mistura
yyyy/MM/dd e dd/MM/yyyy — uma análise de padrão por dígito não distingue
as duas convenções. Sem o terceiro formato no coalesce, aproximadamente
130 pedidos teriam order_date incorretamente nulo. O mesmo conjunto de 3
formatos é aplicado a order_date e promised_date.

Utilização de try_to_date em vez de to_date: no modo ANSI do runtime
atual, to_date lança exceção em formato incompatível em vez de retornar
null, interrompendo o coalesce (comportamento observado com a data
08/11/2024).

In [ ]:
df_cab_step3 = (
    df_cab_step2
    .withColumn(
        "order_date_parsed",
        F.coalesce(
            F.expr("try_to_date(order_date, 'yyyy-MM-dd')"),
            F.expr("try_to_date(order_date, 'yyyy/MM/dd')"),
            F.expr("try_to_date(order_date, 'dd/MM/yyyy')"),
        )
    )
    .withColumn(
        "promised_date_parsed",
        F.coalesce(
            F.expr("try_to_date(promised_date, 'yyyy-MM-dd')"),
            F.expr("try_to_date(promised_date, 'yyyy/MM/dd')"),
            F.expr("try_to_date(promised_date, 'dd/MM/yyyy')"),
        )
    )
    .withColumn("last_update_parsed", F.expr("try_to_timestamp(last_update)"))
)

In [ ]:
df_cab_step4 = df_cab_step3.withColumn(
    "status_order_norm",
    F.when(F.lower(F.trim(F.col("status_order"))) == "faturado", "Faturado")
     .when(F.lower(F.trim(F.col("status_order"))).rlike("em.?separacao"), "Em separação")
     .when(F.lower(F.trim(F.col("status_order"))) == "entregue", "Entregue")
     .when(F.lower(F.trim(F.col("status_order"))) == "cancelado", "Cancelado")
     .otherwise("Não informado")
)

In [ ]:
def parse_valor_brl(col):
    limpo = F.regexp_replace(F.trim(col), ",", ".")
    return F.when(F.upper(F.trim(col)) == "N/A", F.lit(None)).otherwise(limpo.cast("double"))

df_cab_step5 = (
    df_cab_step4
    .withColumn("gross_amount_parsed", parse_valor_brl(F.col("gross_amount")))
    .withColumn("discount_amount_parsed", parse_valor_brl(F.col("discount_amount")))
    .withColumn("net_amount_parsed", parse_valor_brl(F.col("net_amount")))
)

Deduplicação dos 3 order_id conflitantes: cada coluna crítica válida
(data e valor) contribui um ponto para um score de qualidade; mantida a
versão de maior score. Em caso de empate (O00011, duplicata idêntica),
mantida a primeira ocorrência.

In [ ]:
df_cab_step5 = df_cab_step5.withColumn(
    "qualidade_score",
    F.when(F.col("order_date_parsed").isNull(), 0).otherwise(1)
    + F.when(F.col("gross_amount_parsed").isNull(), 0).otherwise(1)
)

w_dedup_cab = Window.partitionBy("order_id").orderBy(
    F.col("qualidade_score").desc(), F.monotonically_increasing_id().asc()
)

df_cab_silver_pre = (
    df_cab_step5
    .withColumn("rk", F.row_number().over(w_dedup_cab))
    .filter(F.col("rk") == 1)
    .drop("rk", "qualidade_score")
)

net_amount nem sempre reconcilia com gross_amount - discount_amount. O
valor não é corrigido; a divergência é sinalizada via flag (tolerância de
R$0,05 para absorver arredondamento). net_amount permanece como
referência de receita, por ser o valor mais próximo do efetivamente
faturado.

In [ ]:
df_cab_silver = df_cab_silver_pre.withColumn(
    "diff_reconciliacao_valor",
    F.round(F.col("net_amount_parsed") - (F.col("gross_amount_parsed") - F.col("discount_amount_parsed")), 2)
).withColumn(
    "flag_valor_inconsistente",
    F.abs(F.col("diff_reconciliacao_valor")) > 0.05
)

df_cab_final = df_cab_silver.select(
    "order_id", "customer_code", "seller_id",
    F.col("order_date_parsed").alias("order_date"),
    F.col("promised_date_parsed").alias("promised_date"),
    F.col("status_order_norm").alias("status_order"),
    F.col("gross_amount_parsed").alias("gross_amount"),
    F.col("discount_amount_parsed").alias("discount_amount"),
    F.col("net_amount_parsed").alias("net_amount"),
    "order_priority", "order_source",
    F.col("last_update_parsed").alias("last_update"),
    "diff_reconciliacao_valor", "flag_valor_inconsistente",
)

(
    df_cab_final.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_pedidos_cabecalho")
)

print("bronze:", df_cab_bronze.count(), "-> silver:", df_cab_final.count())

Tratamento da tabela de itens.

In [ ]:
df_itens_silver = (
    df_itens_bronze
    .withColumn("order_id", F.upper(F.trim(F.col("order_id"))))
    .withColumn("product_code", F.upper(F.trim(F.col("product_code"))))
    .withColumn("quantity", F.col("quantity").cast("double"))
    .withColumn("unit_price", F.regexp_replace(F.trim(F.col("unit_price")), ",", ".").cast("double"))
    .withColumn("total_item", F.col("total_item").cast("double"))
    .withColumn(
        "item_status_norm",
        F.when(F.lower(F.trim(F.col("item_status"))) == "ativo", "Ativo")
         .when(F.lower(F.trim(F.col("item_status"))) == "cancelado", "Cancelado")
         .otherwise("Não informado")
    )
    .withColumn("total_item_calculado", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("flag_total_item_inconsistente", F.abs(F.col("total_item") - F.col("total_item_calculado")) > 0.05)
)

df_itens_final = df_itens_silver.select(
    "order_id", "item_seq", "product_code", "quantity", "unit_price",
    "total_item", "total_item_calculado", "flag_total_item_inconsistente",
    F.col("item_status_norm").alias("item_status"),
)

(
    df_itens_final.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{schema_name}.silver_pedidos_itens")
)

print("bronze:", df_itens_bronze.count(), "-> silver:", df_itens_final.count())

In [ ]:
print("Pedidos sem status informado:", df_cab_final.filter(F.col("status_order") == "Não informado").count())
print("Pedidos com net_amount inconsistente:", df_cab_final.filter(F.col("flag_valor_inconsistente")).count())
print("Itens com total_item inconsistente:", df_itens_final.filter(F.col("flag_total_item_inconsistente")).count())